# Ноутбук 4. Пишем универсальную функцию gsBasis
*(к уроку 09 курса — главное практическое задание!)*

В ноутбуке 3 мы делали шаги руками. Узор такой:

- для столбца номер `j` — вычесть тени на **все** столбцы с номерами меньше `j`;
- потом нормализовать (а если вектор «исчез» — записать нули).

Этот узор идеально ложится на двойной цикл:

```
for j in range(число_столбцов):     # для каждого вектора j
    for i in range(j):              # для каждого ПРЕДЫДУЩЕГО вектора i
        вычесть из столбца j его тень на столбец i
    нормализовать столбец j (или обнулить)
```

Твоя задача ниже — дописать функцию. Но сначала подготовка:

In [ ]:
import numpy as np
import numpy.linalg as la

verySmallNumber = 1e-14     # порог "компьютерного нуля"
print("Готово!")

## ✍️ ГЛАВНОЕ ЗАДАНИЕ: допиши функцию

Замени три `...` на код. Подсказки:

- тень: `B[:, j] = B[:, j] - B[:, j] @ B[:, i] * B[:, i]`— эту строку дарим,
  она уже написана;
- условие: длина столбца `j` больше `verySmallNumber`;
- нормализация: поделить столбец на его длину `la.norm(...)`.

In [ ]:
def gsBasis(A):
    B = np.array(A, dtype=np.float64)       # копия + дробные числа

    for j in range(B.shape[1]):              # B.shape[1] = число столбцов
        for i in range(j):
            # вычитаем тень столбца j на (уже единичный) столбец i:
            B[:, j] = B[:, j] - B[:, j] @ B[:, i] * B[:, i]

        if ...:                               # <- ЗАМЕНИ: длина столбца j > verySmallNumber ?
            B[:, j] = ...                     # <- ЗАМЕНИ: нормализуй столбец j
        else:
            B[:, j] = ...                     # <- ЗАМЕНИ: np.zeros_like(столбец j)

    return B

print("Функция определена. Проверь её следующей ячейкой!")

## Автопроверка

Запусти — она скажет, правильно ли написана функция:

In [ ]:
def proverit_gsBasis():
    try:
        V = np.array([[1, 2, 3],
                      [1, 0, 1],
                      [1, 1, -1]], dtype=np.float64)
        E = gsBasis(V)
        oshibki = []
        for j in range(3):
            if abs(la.norm(E[:, j]) - 1) > 1e-9:
                oshibki.append("длина столбца %d не равна 1" % j)
        for j in range(3):
            for i in range(j):
                if abs(E[:, j] @ E[:, i]) > 1e-9:
                    oshibki.append("столбцы %d и %d не перпендикулярны" % (i, j))
        V2 = np.array([[1, 2], [2, 4]], dtype=np.float64)   # v2 = 2*v1, зависимый
        if la.norm(gsBasis(V2)[:, 1]) > 1e-9:
            oshibki.append("зависимый вектор должен обнуляться")
        if oshibki:
            print("❌ Пока не готово:"); [print("   -", o) for o in oshibki]
        else:
            print("✅ ВСЁ ПРАВИЛЬНО! Твоя gsBasis работает. Результат для примера урока:")
            print(np.round(E, 3))
    except TypeError:
        print("⏳ В функции ещё остались ... - замени их на код")
    except Exception as e:
        print("❌ Функция падает с ошибкой:", e)

proverit_gsBasis()

<details>
<summary>👉 Решение (сначала честно попробуй сам!)</summary>

<pre>
        if la.norm(B[:, j]) > verySmallNumber:
            B[:, j] = B[:, j] / la.norm(B[:, j])
        else:
            B[:, j] = np.zeros_like(B[:, j])
</pre>
</details>

## Когда заработает — испытай её!

In [ ]:
# Тест на 4 векторах в 4D - руками такое считать замучаешься, а функции всё равно:
V4 = np.array([[1, 0, 2, 6],
               [0, 1, 8, 2],
               [2, 8, 3, 1],
               [1, -6, 2, 3]], dtype=np.float64)
try:
    E4 = gsBasis(V4)
    print(np.round(E4, 3))
    print()
    print("проверка E.T @ E (должна быть почти единичная матрица):")
    print(np.round(E4.T @ E4, 6))
except TypeError:
    print("⏳ Сначала допиши функцию gsBasis выше (замени все ...)!")

Про `E.T @ E`: эта запись считает сразу ВСЕ скалярные произведения всех пар
столбцов. Единицы на диагонали = все длины 1; нули вокруг = все пары
перпендикулярны. Одна строка — полная проверка!

## Бонус: считаем независимые направления

In [ ]:
def dimensions(A):
    """Сколько НАСТОЯЩИХ направлений в наборе векторов (ранг)."""
    # после gsBasis каждый столбец имеет длину 1 (настоящий) или 0 (зависимый);
    # складываем длины - получаем число настоящих
    return np.sum(la.norm(gsBasis(A), axis=0))

V_zavisimyy = np.array([[1, 2, 1],
                        [0, 0, 1],
                        [0, 0, 0]], dtype=np.float64)   # v2 = 2*v1 !
try:
    print(np.round(gsBasis(V_zavisimyy), 3))
    print("независимых направлений:", dimensions(V_zavisimyy))
except TypeError:
    print("⏳ Сначала допиши функцию gsBasis выше (замени все ...)!")

---
# 🏋️ Дополнительные задания

1. **Предскажи, потом проверь:** что даст `gsBasis` для матрицы, где все три
   столбца равны `[1, 1, 1]`? Сколько направлений покажет `dimensions`?
2. **Свойство-ловушка:** примени `gsBasis` к результату `gsBasis(V4)` ещё раз.
   Что изменилось? Почему? (подсказка: тени на перпендикулярные оси нулевые)
3. **Сам придумай** матрицу 3×3, у которой `dimensions` = 2, и проверь.

Свободная ячейка для экспериментов:

In [ ]:
# экспериментируй здесь!

<details>
<summary>👉 Ответы на дополнительные</summary>

<pre>
1. Первый столбец нормализуется, остальные обнулятся; dimensions = 1.
2. Ничего не изменится: набор уже ортонормированный, все тени нулевые,
   все длины уже 1. Хороший тест на правильность функции!
3. Например: столбцы [1,0,0], [0,1,0], [1,1,0] - третий равен сумме первых двух.
</pre>
</details>

---
Дальше — **ноутбук 5**: применяем gsBasis к отражению в наклонном зеркале.